<a href="https://colab.research.google.com/github/pataneniruthwik/mini-project-2/blob/main/MINORPROJECT2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
# Import required libraries
import pandas as pd
import numpy as np

# Import file uploader
from google.colab import files

# Upload file
uploaded = files.upload()



import pandas as pd
df=pd.read_csv("Data set for DADS June.csv")
df
df.info()
print(df["Date"].unique())
# Convert all date formats into one standard datetime format
df["Date"] = pd.to_datetime(
    df["Date"],

    dayfirst=True,
     format="mixed",
    errors="coerce",
)
df.head(20)
print(df["Date"].isna().sum())
df["Type"].unique()
# Convert everything to lowercase
df["Type"] = df["Type"].str.lower()

# Standardize the values
df["Type"] = df["Type"].replace({
    "dr": "debit",
    "cr": "credit"
})

df["Amount"].unique()
df["Amount"] = (
    df["Amount"]
    .astype(str)
    .str.replace("₹", "", regex=False)
    .str.replace("Rs.", "", regex=False)
    .str.replace("Rs", "", regex=False)
    .str.replace(",", "", regex=False)
    .str.strip()
)

df["Amount"] = pd.to_numeric(df["Amount"], errors="coerce")

df.head(20)

print(df["Amount"].isna().sum())

print(df["Amount"].dtype)
df["Mode"].unique()

# Remove duplicate rows
df = df.drop_duplicates()

print(df.duplicated().sum())
df.head(20)

df["Description"].unique()

# -----------------------------
# Feature 2 : Vendor Extraction
# -----------------------------

# Dictionary containing vendor keywords
vendor_dict = {

    # Food Delivery
    "SWIGGY": "Swiggy",
    "BUNDL": "Swiggy",
    "ZOMATO": "Zomato",

    # Quick Commerce
    "ZEPTO": "Zepto",
    "BLINKIT": "Blinkit",
    "INSTAMART": "Instamart",
    "BIGBASKET": "BigBasket",
    "DMART": "DMart",

    # Shopping
    "AMAZON": "Amazon",
    "AMAZON PAY": "Amazon Pay",
    "AMZN": "Amazon",
    "FLIPKART": "Flipkart",
    "FKART": "Flipkart",
    "MYNTRA": "Myntra",

    # Transport
    "UBER": "Uber",
    "OLA": "Ola",
    "ANI": "Ola",
    "RAPIDO": "Rapido",
    "BMTC": "BMTC",

    # Fuel
    "HP": "HP Petrol",
    "BPCL": "BPCL Petrol",
    "INDIAN OIL": "Indian Oil",
    "IOCL": "Indian Oil",

    # Entertainment
    "NETFLIX": "Netflix",
    "SPOTIFY": "Spotify",
    "HOTSTAR": "Disney+ Hotstar",
    "PRIME VIDEO": "Prime Video",
    "AMAZON PRIME": "Amazon Prime",
    "BOOKMYSHOW": "BookMyShow",
    "BMS": "BookMyShow",

    # Coffee / Food
    "STARBUCKS": "Starbucks",
    "CAFE COFFEE DAY": "Cafe Coffee Day",
    "CCD": "Cafe Coffee Day",
    "THIRD WAVE": "Third Wave Coffee",

    # Investment
    "GROWW": "Groww",
    "ZERODHA": "Zerodha",
    "COIN": "Zerodha Coin",

    # Bills
    "BESCOM": "BESCOM",
    "JIO": "Jio",
    "AIRTEL": "Airtel",
    "VODAFONE": "Vodafone Idea",
    "VI": "Vodafone Idea",

    # Travel
    "IRCTC": "IRCTC",

    # ATM
    "ATM-WDL": "ATM Withdrawal",

    # Salary
    "SALARY": "Salary",

    # Recharge
    "RECHARGE": "Mobile Recharge"
}


# Function to extract vendor name
def extract_vendor(description):

    # Convert description into uppercase
    description = description.upper()

    # Check every keyword in the dictionary
    for keyword, vendor in vendor_dict.items():

        # If keyword is found inside description
        if keyword in description:
            return vendor

    # If no keyword matches
    return "Others"


# Create a new Vendor column
df["Vendor"] = df["Description"].apply(extract_vendor)


# Display the result
print(df[["Description", "Vendor"]].head(20))

category_dict = {

    "Swiggy": "Food Delivery",
    "Zomato": "Food Delivery",

    "Zepto": "Quick Commerce",
    "Blinkit": "Quick Commerce",
    "BigBasket": "Quick Commerce",
    "Instamart": "Quick Commerce",

    "Amazon": "Shopping",
    "Flipkart": "Shopping",
    "Myntra": "Shopping",

    "Uber": "Transport",
    "Ola": "Transport",
    "Rapido": "Transport",
    "BMTC": "Transport",

    "Netflix": "Entertainment",
    "Spotify": "Entertainment",
    "BookMyShow": "Entertainment",

    "Groww": "Investment",
    "Zerodha": "Investment",

    "Jio": "Bills",
    "Airtel": "Bills",
    "BESCOM": "Bills",

    "HP Petrol": "Fuel",
    "BPCL Petrol": "Fuel",
    "Indian Oil": "Fuel",

    "Salary": "Income",

    "ATM Withdrawal": "Cash Withdrawal",

    "Personal UPI Transfer": "Money Transfer",

    "Others": "Others"
}
def assign_category(vendor):

    if vendor in category_dict:
        return category_dict[vendor]

    return "Others"
df["Category"] = df["Vendor"].apply(assign_category)
df[["Vendor", "Category"]].head(20)
print("========== SpendDNA Overview ==========")

# Total Transactions
total_transactions = len(df)
print("Total Transactions :", total_transactions)

# Total Debit
total_debit = df[df["Type"] == "debit"]["Amount"].sum()
print("Total Debit :", total_debit)

# Total Credit
total_credit = df[df["Type"] == "credit"]["Amount"].sum()
print("Total Credit :", total_credit)

# Net Balance
net_balance = total_credit - total_debit
print("Net Balance :", net_balance)

# Top Vendors
print("\nTop Vendors")
print(df.groupby("Vendor")["Amount"].sum().sort_values(ascending=False).head())

# Top Categories
print("\nTop Categories")
print(df.groupby("Category")["Amount"].sum().sort_values(ascending=False))

# ==========================================
# FEATURE 5 : Monthly Trend Analysis
# ==========================================

# Create Month Column
df["Month"] = df["Date"].dt.month_name()

# Monthly Spending
monthly_spending = df.groupby("Month")["Amount"].sum()

print("========== Monthly Spending ==========")
print(monthly_spending)

print("\nHighest Spending Month :", monthly_spending.idxmax())
print("Lowest Spending Month :", monthly_spending.idxmin())

# Category-wise Monthly Spending
monthly_category = df.pivot_table(
    values="Amount",
    index="Month",
    columns="Category",
    aggfunc="sum",
    fill_value=0
)

print("\n========== Category Wise Monthly Spending ==========")
print(monthly_category)



# ==========================================
# FEATURE 6 : Time Analysis
# ==========================================

# Convert Time into datetime
df["Time"] = pd.to_datetime(df["Time"], format="%H:%M")

# Extract Hour
df["Hour"] = df["Time"].dt.hour


# Function to classify time
def time_period(hour):

    if hour < 12:
        return "Morning"

    elif hour < 17:
        return "Afternoon"

    elif hour < 21:
        return "Evening"

    else:
        return "Night"


# Create Time Period Column
df["Time Period"] = df["Hour"].apply(time_period)

print("\n========== Time Analysis ==========")
print(df["Time Period"].value_counts())



# ==========================================
# FEATURE 7 : Anomaly Detection
# ==========================================

# Mean Amount
mean = df["Amount"].mean()

# Standard Deviation
std = df["Amount"].std()

# Calculate Z Score
df["Z Score"] = (df["Amount"] - mean) / std

# Find Anomalies
anomalies = df[df["Z Score"] > 3]

print("\n========== Anomalies ==========")
print(anomalies[["Date","Vendor","Amount","Z Score"]])



# ==========================================
# FEATURE 8 : Spending Personality
# ==========================================

print("\n========== Spending Personality ==========")

category_total = df.groupby("Category")["Amount"].sum()

# Foodie
if "Food Delivery" in category_total.index:
    if category_total["Food Delivery"] > 5000:
        print("🍔 Foodie")

# Shopaholic
if "Shopping" in category_total.index:
    if category_total["Shopping"] > 10000:
        print("🛍️ Shopaholic")

# Investor
if "Investment" in category_total.index:
    if category_total["Investment"] > 5000:
        print("📈 Investor")

# Late Night Spender
night_transactions = len(df[df["Time Period"] == "Night"])

if night_transactions > 30:
    print("🌙 Late Night Spender")



# ==========================================
# FINAL REPORT
# ==========================================

print("\n=======================================")
print("         SpendDNA Final Report")
print("=======================================")

print("Total Transactions :", len(df))
print("Total Debit :", total_debit)
print("Total Credit :", total_credit)
print("Net Balance :", net_balance)

print("\nTop 5 Vendors")
print(df.groupby("Vendor")["Amount"].sum().sort_values(ascending=False).head())

print("\nTop Categories")
print(df.groupby("Category")["Amount"].sum().sort_values(ascending=False))

print("\nHighest Spending Month :", monthly_spending.idxmax())
print("Lowest Spending Month :", monthly_spending.idxmin())

print("\nTransactions by Time Period")
print(df["Time Period"].value_counts())

print("\nNumber of Anomalies :", len(anomalies))

print("\n============= END OF REPORT =============")

Saving Data set for DADS June.csv to Data set for DADS June (2).csv
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1328 entries, 0 to 1327
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Date         1328 non-null   object 
 1   Time         1328 non-null   object 
 2   Description  1328 non-null   object 
 3   Type         1328 non-null   object 
 4   Amount       1328 non-null   object 
 5   Balance      1328 non-null   float64
 6   Mode         1328 non-null   object 
 7   Ref          1328 non-null   object 
dtypes: float64(1), object(7)
memory usage: 83.1+ KB
['2024-01-01' '01-Jan-24' '01 Jan 2024' '2024-01-02' '02/01/24'
 '02-Jan-24' '02 Jan 2024' '03-Jan-24' '03/01/24' '2024-01-03' '04-Jan-24'
 '2024-01-04' '04 Jan 2024' '04/01/24' '05-Jan-24' '05/01/24' '2024-01-05'
 '06 Jan 2024' '06/01/24' '06-Jan-24' '07/01/24' '2024-01-07'
 '07 Jan 2024' '08/01/24' '2024-01-08' '08-Jan-24' '08 Jan 2024'
 '09 Jan 202